# Train, Evaluate, and Track Baseline vs Challenger Models

In [0]:
import mlflow
import mlflow.spark
import os
import mlflow
import mlflow.spark

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, TrainValidationSplit
from mlflow.models.signature import infer_signature

In [0]:
# Build Unity Catalog Volume for MLflow (temporary)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.insurance_claim.mlflow_tmp")
tmp_vol_path = "/Volumes/workspace/insurance_claim/mlflow_tmp"

# Set the environment variable for Spark ML caching
# This is required for CrossValidator on UC Shared/Serverless clusters
os.environ["SPARKML_TEMP_DFS_PATH"] = tmp_vol_path

# Load data from Unity Catalog
train_data = spark.table("workspace.insurance_claim.train_set")
test_data = spark.table("workspace.insurance_claim.test_set")

# Prepare the evaluator
evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)

### Baseline Model (Logistic Regression)

In [0]:
# =============================================================================
# 🟢 RUN 1: Baseline Model (Logistic Regression)
# =============================================================================

with mlflow.start_run(run_name="Baseline_LogisticRegression"):
    print("Training Baseline Model...")
    lr = LogisticRegression(featuresCol="features", labelCol="is_fraud", maxIter=10)
    lr_model = lr.fit(train_data)
    
    # Make predictions and evaluate
    lr_predictions = lr_model.transform(test_data)
    lr_auc = evaluator.evaluate(lr_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_metric("ROC_AUC", lr_auc)
    
    # add variable: dfs_tmpdir to point at UC Volume
    mlflow.spark.log_model(lr_model, "baseline_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Baseline ROC AUC: {lr_auc:.4f}")

### Challenger Model (Random Forest)

In [0]:
# =============================================================================
# 🔵 RUN 2/1: Challenger Model (Random Forest)
# =============================================================================

with mlflow.start_run(run_name="Challenger_RandomForest"):
    print("Training Challenger Model...")
    rf = RandomForestClassifier(
    featuresCol="features", 
    labelCol="is_fraud", 
    numTrees=50, 
    maxDepth=5,
    maxBins=15000
    )
    rf_model = rf.fit(train_data)
    
    # Make predictions and evaluate
    rf_predictions = rf_model.transform(test_data)
    rf_auc = evaluator.evaluate(rf_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_metric("ROC_AUC", rf_auc)
    
    # add variable: dfs_tmpdir to pint at UC Volume
    mlflow.spark.log_model(rf_model, "challenger_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Challenger ROC AUC: {rf_auc:.4f}")

In [0]:
# =============================================================================
# 🔵 RUN 2/2: Challenger Model (Random Forest)
# =============================================================================

with mlflow.start_run(run_name="Challenger_RandomForest"):
    print("Training Challenger Model...")
    
    # ADDED: maxBins=15000 to handle high cardinality in categorical features
    rf = RandomForestClassifier(
        featuresCol="features", 
        labelCol="is_fraud", 
        numTrees=50, 
        maxDepth=5,
        maxBins=15000 
    )
    rf_model = rf.fit(train_data)
    
    # Make predictions on the test set and evaluate the model
    rf_predictions = rf_model.transform(test_data)
    rf_auc = evaluator.evaluate(rf_predictions)
    
    # Log model parameters to MLflow for tracking and comparison
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("maxBins", 15000) # ADDED: Good practice to log the modified maxBins
    
    # Log the evaluation metric (ROC AUC)
    mlflow.log_metric("ROC_AUC", rf_auc)
    
    # ADDED: dfs_tmpdir=tmp_vol_path to safely log the model via Unity Catalog Volume 
    # (Bypasses restrictions on Databricks Shared/Serverless clusters)
    mlflow.spark.log_model(rf_model, "challenger_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Challenger ROC AUC: {rf_auc:.4f}")

### Updated Strategy
Goal: Use Default Random Forest as Baseline, and GBT as Challenger

In [0]:
# =============================================================================
# 🟢 RUN 3: NEW Baseline Model (Default Random Forest)
# =============================================================================
with mlflow.start_run(run_name="Baseline_Default_RandomForest"):
    print("Training Baseline Model (Default RF)...")
    
    # Using mostly default parameters (except maxBins for your specific data)
    rf_default = RandomForestClassifier(
        featuresCol="features", 
        labelCol="is_fraud",
        maxBins=15000 
        # Not setting numTrees or maxDepth here, letting Spark use its defaults
    )
    rf_baseline_model = rf_default.fit(train_data)
    
    # Evaluate
    baseline_predictions = rf_baseline_model.transform(test_data)
    baseline_auc = evaluator.evaluate(baseline_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Random Forest (Default)")
    mlflow.log_param("maxBins", 15000)
    mlflow.log_metric("ROC_AUC", baseline_auc)
    
    # Log model to UC Volume
    mlflow.spark.log_model(rf_baseline_model, "baseline_model", dfs_tmpdir=tmp_vol_path)
    print(f"✅ Baseline ROC AUC: {baseline_auc:.4f}")

In [0]:
# =============================================================================
# 🔵 RUN 4/1: Gradient Boosted Trees
# =============================================================================

with mlflow.start_run(run_name="Challenger_GBTClassifier_With_Signature"):
    print("Training Challenger Model (GBT)...")
    
    # 1. Train Model
    gbt = GBTClassifier(
        featuresCol="features", 
        labelCol="is_fraud", 
        maxIter=20,     
        maxDepth=5,     
        maxBins=15000
    )
    gbt_model = gbt.fit(train_data)
    
    # 2. Evaluate
    gbt_predictions = gbt_model.transform(test_data)
    gbt_auc = evaluator.evaluate(gbt_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Gradient Boosted Trees")
    mlflow.log_param("maxIter", 20)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("maxBins", 15000)
    mlflow.log_metric("ROC_AUC", gbt_auc)
    
    # =============================================================================
    # 🌟 NEW: Create Model Signature
    # Tells Unity Catalog the exact schema of inputs and outputs
    # =============================================================================
    signature = infer_signature(
        model_input=train_data.select("features"), 
        model_output=gbt_predictions.select("prediction")
    )
    
    # =============================================================================
    # 🌟 MODIFIED: Log model WITH signature attached
    # =============================================================================
    mlflow.spark.log_model(
        spark_model=gbt_model, 
        artifact_path="challenger_model", 
        signature=signature,
        dfs_tmpdir=tmp_vol_path
    )
    print(f"✅ Challenger ROC AUC: {gbt_auc:.4f}")

### Hyperparameter Tuning for GBT Classifier using CrossValidator

In [0]:
# =============================================================================
# 🚀 RUN 4/2: Hyperparameter Tuning for GBT
# =============================================================================

with mlflow.start_run(run_name="Tuned_GBTClassifier_TVS"):
    print("Starting Memory-Friendly Tuning for GBT...")
    
    # Initialize Base GBT 
    gbt = GBTClassifier(featuresCol="features", labelCol="is_fraud", maxBins=15000)
    
    # Create Parameter Grid (8 combinations)
    paramGrid = ParamGridBuilder() \
        .addGrid(gbt.maxIter, [20, 50]) \
        .addGrid(gbt.maxDepth, [4, 6]) \
        .addGrid(gbt.stepSize, [0.1, 0.05]) \
        .build()
    
    # 🌟 THE FIX: Use TrainValidationSplit instead of CrossValidator
    # This evaluates each parameter combination only ONCE (using 80% train / 20% validation)
    # It trains 8 models total instead of 24, bypassing the 1GB cache limit!
    tvs = TrainValidationSplit(
        estimator=gbt,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        trainRatio=0.8  # 80% of train_data used for training, 20% for validation during tuning
        # Removed parallelism to keep memory footprint minimal
    )
    
    # Train Models (This will be much faster and consume less memory)
    tvs_model = tvs.fit(train_data)
    best_gbt = tvs_model.bestModel
    
    # Evaluate the BEST Model on the Hold-out Test Set
    best_predictions = best_gbt.transform(test_data)
    best_auc = evaluator.evaluate(best_predictions)
    
    # Log parameters of the winning model to MLflow
    mlflow.log_param("model_type", "Tuned GBT (TVS)")
    mlflow.log_param("best_maxIter", best_gbt.getMaxIter())
    mlflow.log_param("best_maxDepth", best_gbt.getMaxDepth())
    mlflow.log_param("best_stepSize", best_gbt.getStepSize())
    mlflow.log_metric("ROC_AUC", best_auc)
    
    # Log the best model safely
    mlflow.spark.log_model(best_gbt, "tuned_gbt_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Tuned GBT ROC AUC: {best_auc:.4f}")
    print(f"🏆 Winning Parameters: maxIter={best_gbt.getMaxIter()}, maxDepth={best_gbt.getMaxDepth()}, stepSize={best_gbt.getStepSize()}")

### Register the Winning GBT Model to Unity Catalog

In [0]:
## Register the Winning GBT Model to Unity ![Catalog](![path](path))

# Replace with your actual Run ID from the MLflow UI (Experiment tab)
# You can find this 32-character string in the Run details page
run_id = "8e82a5a0f9084144a3550254772e9371" 

# Define the model name using Unity Catalog structure (catalog.schema.model_name)
model_name = "workspace.insurance_claim.fraud_gbt_model"

# Define the URI path to the logged model artifact
model_uri = f"runs:/{run_id}/challenger_model"

# Register the model into Unity Catalog Model Registry
try:
    registered_model = mlflow.register_model(
        model_uri=model_uri, 
        name=model_name
    )
    print(f"✅ Successfully registered model: {registered_model.name} (Version: {registered_model.version})")
except Exception as e:
    print(f"❌ Error registering model: {e}")

### Extract and visualize Feature Importance from the Best GBT Model

In [0]:
print("🔍 Extracting Feature Importance from the winning model...")

# 1. Extract the raw importance scores from the best GBT model
importances = best_gbt.featureImportances.toArray()

# 2. Extract feature names directly from the 'features' column metadata
# (Spark's VectorAssembler automatically stores the original column names here)
feature_metadata = train_data.schema["features"].metadata.get("ml_attr", {}).get("attrs", {})

feature_list = []
# Loop through numeric, nominal (categorical), and binary metadata to get names
for attr_type in ["numeric", "nominal", "binary"]:
    if attr_type in feature_metadata:
        for attr in feature_metadata[attr_type]:
            feature_list.append({"index": attr["idx"], "name": attr["name"]})

# 3. Combine names and scores into a Pandas DataFrame
if len(feature_list) > 0:
    df_importance = pd.DataFrame(feature_list).sort_values("index").reset_index(drop=True)
    df_importance["importance"] = importances
else:
    # Fallback in case metadata was stripped during data prep
    print("⚠️ Warning: Metadata not found. Using feature index instead.")
    df_importance = pd.DataFrame({
        "name": [f"Feature_{i}" for i in range(len(importances))],
        "importance": importances
    })

# 4. Sort to find the Top 15 most important features
top_15_features = df_importance.sort_values(by="importance", ascending=False).head(15)

# 5. Display as a Table
print("\n🎯 Top 15 Most Important Features:")
print(top_15_features.to_string(index=False))

# 6. Visualize with a Horizontal Bar Chart
plt.figure(figsize=(10, 6))
# Using seaborn for a clean, professional look
sns.barplot(x="importance", y="name", data=top_15_features, palette="viridis")

plt.title("Top 15 Drivers of Insurance Fraud (GBT Model)", fontsize=14, pad=15)
plt.xlabel("Importance Score (Higher = Stronger Predictor)", fontsize=12)
plt.ylabel("Feature Name", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()

# Show the plot directly in Databricks
plt.show()